In [1]:
# Data cleaning pipeline
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
# Let's read the raw data
import os
BASE_DIR = os.getcwd()
PARENT_DIR = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PARENT_DIR, 'data')
DATA_DIR

'd:\\python\\AI\\pynb\\data'

In [3]:
# identify the encoding
hc_report:str = f"{DATA_DIR}\\hc_report.csv"
# import chardet
# with open(hc_report, "rb") as f:
#     raw = f.read(100000)

# result:dict = chardet.detect(raw)
# print(result)
# encoding = result.get('encoding')

In [15]:
# laod raw data in df
df = pd.read_csv(hc_report, encoding='cp1252')
df.dtypes

Login ID(BACG)             str
Worker ID                  str
Worker                     str
GDCE ID(BACG)              str
Ariba Contract ID(BACG)    str
                          ... 
Rejection Comments.1       str
Cost Center Code           str
Primary Cost Center        str
Cost Center                str
Resource Location          str
Length: 72, dtype: object

In [ ]:
# Step 1 to identify data quality issues
def data_quality_check(*, df: pd.DataFrame,):
    quality_report ={
        'total_records': len(df),
        'duplicate_rows': df.duplicated().sum(),
        'missing_values': df.isnull().sum().to_dict(),
        'invalid_emails': df[df['Email'].str.contains('@', na=False)].shape[0],
        'invalid_supervisor_email': df[-df['Resource BAC Supervisor E-mail Address(BACG)'].str.contains('@', na=False)].shape[0]
    }

    return quality_report

In [ ]:
# Step 2: Remove duplicates
df_clean = df.copy()
initial_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Worker ID', 'Email'], keep='first')
duplicates_removed = initial_count - len(df_clean)
duplicates_removed
# df[df.duplicated(subset=['Worker ID', 'Email'])] # Use this to see the duplicate records

2

In [ ]:
df.columns

Index(['Login ID(BACG)', 'Worker ID', 'Worker', 'GDCE ID(BACG)',
       'Ariba Contract ID(BACG)', 'SOW Worker Role', 'Bill Rate [ST/Hr]',
       'Billing Type(BACG)', 'Worker Start Date', 'Worker End Date',
       'Worker Status', 'Email', 'Worker: Worker Supervisor',
       'Resource BAC Supervisor E-mail Address(BACG)', 'Skill 1(BACG)',
       'Specify Programming Language for OTHERS or Mention N/A(BACG)',
       'Skill 1 Experience (in years)(BACG)', 'Skill 1 Certification(BACG)',
       'Skill 1 Year of Certification(BACG)', 'Person ID(BACG)',
       'Resource Mailcode(BACG)', 'Original Start Date(BACG)',
       'Worker Closed Date', 'Statement of Work ID', 'Statement of Work',
       'Resource Background Check Completion Date(BACG)',
       'Experience Category(BACG)', 'Worker Registration Date',
       'Is Resource Employed with a Subcontractor?(BACG)',
       'If YES, Name the Subcontractor that Employs the Worker?(BACG)',
       'Work Order ID', 'Resource Type Requested(BACG)'

In [ ]:
# step 3: Standardize tex formatting
df_clean['Worker'] = df_clean['Worker'].str.strip().str.title()
df_clean['Email'] = df_clean['Email'].str.strip().str.lower()

In [ ]:
# Step 4: Handle missing values
df_clean['Experience Category(BACG)'] = df_clean['Experience Category(BACG)'].fillna('Not Provided')

In [ ]:
# Step 5 filter invalid records
df_clean = df_clean[df_clean['Email'].notna()]

In [ ]:
# Step 6: Enrich with calculation fields
df_clean['common_id'] = df_clean['Worker ID'] + '_'+df_clean['Login ID(BACG)']
df_clean['Worker Start Date'] = pd.to_datetime(df_clean['Worker Start Date'], errors='coerce')
df_clean['day_since_woker_startdate'] = (datetime.now() - df_clean['Worker Start Date']).dt.days

In [ ]:
df_clean['day_since_woker_startdate'].head()

0    1115
1     735
2     413
3     627
4     690
Name: day_since_woker_startdate, dtype: int64

In [ ]:
# Step 7: Add data quality flag
df_clean['data_quality_flag'] = np.where(
    (df_clean['Experience Category(BACG)'] == 'Not Provided'),
    'Incomplete',
    'Complete'
)

In [ ]:
df_clean['data_quality_flag'].value_counts()

data_quality_flag
Incomplete    227
Complete      172
Name: count, dtype: int64

In [ ]:
df_clean['Experience Category(BACG)']

0        Specialist - Seven to Less than Ten Years
1                       Basic - Less than one year
2                                     Not Provided
3                                     Not Provided
4                                     Not Provided
                          ...                     
394            Specialist - Greater than Ten years
395            Specialist - Greater than Ten years
396                                   Not Provided
397    Specialist - Three to less than seven years
398         Basic - Three to less than seven years
Name: Experience Category(BACG), Length: 399, dtype: str